# Job Radar InHire - Data Analyst

Scanner automático de vagas em empresas que usam InHire.

Funcionalidades:

- Descobre empresas InHire automaticamente
- Extrai links diretos de vagas
- Ignora acentos
- Ignora case sensitive
- Filtra vagas de dados
- Salva CSV
- Preparado para Telegram


In [ ]:
import requests
import pandas as pd
import re
import time
import unicodedata


## Normalização de texto (remove acentos)

In [ ]:
def normalize(text):
    text = text.lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text

## Palavras-chave de vagas de dados

In [ ]:
KEYWORDS = [
    "data",
    "dados",
    "analytics",
    "analyst",
    "bi",
    "machine learning",
    "cientista",
    "engenheiro de dados",
    "data engineer",
    "data scientist"
]

## Descobrir empresas InHire automaticamente

In [ ]:
def discover_inhire_companies():

    print("Buscando empresas InHire...")

    url = "https://crt.sh/?q=%25.inhire.app&output=json"

    r = requests.get(url)

    data = r.json()

    domains = set()

    for entry in data:

        name = entry['name_value']

        if '.inhire.app' in name:

            company = name.split('.inhire.app')[0]

            domains.add(company)

    companies = sorted(domains)

    print("Empresas encontradas:", len(companies))

    return companies


COMPANIES = discover_inhire_companies()

print(COMPANIES[:20])

## Extrair links de vagas

In [ ]:
def extract_jobs(company):

    url = f"https://{company}.inhire.app/vagas"

    jobs = []

    try:

        r = requests.get(url, timeout=10)

        if r.status_code != 200:
            return jobs

        html = r.text

        pattern = rf"https://{company}\.inhire\.app/vagas/[a-z0-9\-]+/[a-z0-9\-]+"

        links = re.findall(pattern, html)

        for link in links:

            title = link.split("/")[-1]

            jobs.append({
                "empresa": company,
                "titulo": title.replace("-"," "),
                "link": link
            })

    except Exception as e:

        print("Erro em", company)

    return jobs

## Verificar se é vaga de dados

In [ ]:
def is_data_job(title):

    title = normalize(title)

    for k in KEYWORDS:
        if normalize(k) in title:
            return True

    return False

## Executar radar

In [ ]:
all_jobs = []

for company in COMPANIES:

    print("Escaneando:", company)

    jobs = extract_jobs(company)

    all_jobs.extend(jobs)

    time.sleep(0.5)

print("\nTotal vagas encontradas:", len(all_jobs))

data_jobs = [j for j in all_jobs if is_data_job(j['titulo'])]

print("Vagas de dados encontradas:", len(data_jobs))


for job in data_jobs:

    print("\n🚨 Nova vaga")
    print("Empresa:", job['empresa'])
    print("Cargo:", job['titulo'])
    print("Link:", job['link'])


df = pd.DataFrame(data_jobs)

df.to_csv("vagas_data_analyst.csv", index=False)

print("\nArquivo salvo: vagas_data_analyst.csv")